# 02. 데이터 적재와 전처리

2023–2025년 노선 여객·운항 Excel 6개와 2026년 1–7월 노선 Excel 2개를 같은 규칙으로 처리한다. 여객은 예측 목표, 운항 편수와 출발 공항 수는 모델 입력이다. 파일 해시로 중복 적재를 막고, raw 테이블과 뷰를 분리해 원문을 보존한다. 국내선 형태의 행과 합계 행을 식별한 뒤 국제선 목적지×월 패널을 만든다.

In [1]:
from pathlib import Path
import os, sys
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL
from IPython.display import display
ROOT=Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'scripts'))
import portfolio_data as pdx
load_dotenv(ROOT/'.env')
SOURCE_ROOT=ROOT/'data/raw/airportal'; HOLDOUT_ROOT=ROOT/'data/raw_2026/airportal'
assert SOURCE_ROOT.exists() and HOLDOUT_ROOT.exists()
db_name=os.environ['DB_NAME']; assert db_name.replace('_','').isalnum()
server=create_engine(URL.create('mysql+pymysql',host=os.environ['DB_HOST'],port=int(os.getenv('DB_PORT','3306')),username=os.environ['DB_USER'],password=os.environ['DB_PASSWORD']),future=True)
with server.begin() as connection:
    connection.execute(text(f'CREATE DATABASE IF NOT EXISTS `{db_name}` CHARACTER SET utf8mb4 COLLATE utf8mb4_unicode_ci'))
engine=pdx.get_engine()
schema=(pdx.sql_root()/'00_elt_pipeline.sql').read_text(encoding='utf-8')
schema='\n'.join(line for line in schema.splitlines() if not line.strip().startswith('--'))
with engine.begin() as connection:
    for statement in schema.split(';'):
        if statement.strip(): connection.execute(text(statement))
print('Raw schema ready')

Raw schema ready


## 1. Excel 카탈로그와 원본 적재

In [2]:
DATASETS = [
    {'target': 'raw_route_passenger', 'pattern': 'transport_statistics/route/route_passenger_*.xlsx',
     'columns': ['노선1', '노선2', '년월', '여객(명)'],
     'rename': {'노선1': 'origin_airport_raw', '노선2': 'destination_airport_raw', '년월': 'period_raw', '여객(명)': 'passengers'}},
    {'target': 'raw_route_operations', 'pattern': 'transport_statistics/route/route_operations_*.xlsx',
     'columns': ['노선1', '노선2', '년월', '운항(편)'],
     'rename': {'노선1': 'origin_airport_raw', '노선2': 'destination_airport_raw', '년월': 'period_raw', '운항(편)': 'operations'}},
]
manifest=pdx.source_manifest(SOURCE_ROOT,DATASETS)
display(manifest)
assert manifest.file_count.sum()==6
loaded=pdx.load_files(engine,SOURCE_ROOT,DATASETS,'destination_demand_2023_2025')
display(loaded.groupby(['target_table','result'],as_index=False).rows.sum().head())

,target_table,file_count
0,raw_route_passenger,3
1,raw_route_operations,3


,target_table,result,rows
0,raw_route_operations,skipped,0
1,raw_route_passenger,skipped,0


In [3]:
source_counts=[]
for dataset in DATASETS:
    n=sum(len(pdx.prepare_frame(path,dataset)) for path in SOURCE_ROOT.glob(dataset['pattern']))
    db=int(pd.read_sql(text(f"SELECT COUNT(*) AS n FROM {dataset['target']}"),engine).n.iat[0])
    source_counts.append({'table':dataset['target'],'Excel_rows':n,'DB_rows':db,'difference':db-n})
source_parity=pd.DataFrame(source_counts)
display(source_parity)
assert source_parity.difference.eq(0).all()

,table,Excel_rows,DB_rows,difference
0,raw_route_passenger,9670,9670,0
1,raw_route_operations,9670,9670,0


In [4]:
raw_sql=pdx.named_sql('01_raw_overview.sql')
for name in ['route_raw_counts','route_raw_columns','route_raw_quality']:
    result=pdx.query(engine,raw_sql,name)
    print(name, result.shape); display(result.head())
raw_totals=pdx.query(engine,raw_sql,'route_raw_total_parity')
assert raw_totals[['reported_total','detail_total']].notna().all().all()
assert len(raw_totals)==72 and raw_totals['difference'].eq(0).all()
print('합계 행과 세부 행의 월별 차이 0:',len(raw_totals),'행')
display(raw_totals.head())
display(pd.read_sql(text('SELECT * FROM raw_route_passenger LIMIT 5'),engine))

route_raw_counts (2, 2)


,table_name,row_count
0,raw_route_passenger,9670
1,raw_route_operations,9670


route_raw_columns (8, 5)


,TABLE_NAME,ORDINAL_POSITION,COLUMN_NAME,COLUMN_TYPE,IS_NULLABLE
0,raw_route_operations,1,origin_airport_raw,varchar(100),NO
1,raw_route_operations,2,destination_airport_raw,varchar(100),YES
2,raw_route_operations,3,period_raw,varchar(20),NO
3,raw_route_operations,4,operations,bigint,YES
4,raw_route_passenger,1,origin_airport_raw,varchar(100),NO


route_raw_quality (2, 6)


,metric,rows_count,periods,null_values,zero_values,aggregate_rows
0,passengers,9670,36,0.0,1107.0,36.0
1,operations,9670,36,0.0,241.0,36.0


합계 행과 세부 행의 월별 차이 0: 72 행


,metric,period_raw,reported_total,detail_total,difference
0,operations,2023년 01월,13632,13632.0,0.0
1,operations,2023년 02월,12937,12937.0,0.0
2,operations,2023년 03월,14407,14407.0,0.0
3,operations,2023년 04월,15231,15231.0,0.0
4,operations,2023년 05월,16782,16782.0,0.0


,origin_airport_raw,destination_airport_raw,period_raw,passengers
0,전체 합계,None,2023년 01월,2403433
1,전체 합계,None,2023년 02월,2212589
2,전체 합계,None,2023년 03월,2256529
3,전체 합계,None,2023년 04월,2412373
4,전체 합계,None,2023년 05월,2622755


노선 원본의 `전체 합계`는 세부 행 합계와 맞는지 확인한 뒤 분석 집계에서 제외한다. 출발·도착 공항명과 년월은 raw에 그대로 두고 뷰에서만 표준화한다. 결측이나 파싱 실패를 임의 수요 0으로 바꾸지 않는다.

## 2. 국제선 범위·목적지 표준화·분석 뷰

In [5]:
prep_sql=pdx.named_sql('02_preprocessing_views.sql')
pdx.apply_ddl(engine,prep_sql,['create_v_domestic_airport_reference','create_v_route_monthly'])
for name in ['route_view_row_parity','route_view_total_parity','route_scope_summary',
             'destination_name_collisions','monthly_period_parse_check','route_scope_monthly_comparison']:
    result=pdx.query(engine,prep_sql,name)
    print(name,result.shape);display(result.head())
row_parity=pdx.query(engine,prep_sql,'route_view_row_parity').iloc[0]
assert row_parity.passenger_detail_rows==row_parity.passenger_view_rows
assert row_parity.operation_detail_rows==row_parity.operation_view_rows
view_parity=pdx.query(engine,prep_sql,'route_view_total_parity')
assert len(view_parity)==72 and view_parity['difference'].eq(0).all()
assert int(pdx.query(engine,prep_sql,'monthly_period_parse_check').invalid_period_rows.iat[0])==0
display(pd.read_sql(text('SELECT * FROM v_route_monthly LIMIT 5'),engine))

route_view_row_parity (1, 4)


,passenger_detail_rows,passenger_view_rows,operation_detail_rows,operation_view_rows
0,9634,9634,9634,9634


route_view_total_parity (72, 5)


,metric,period_raw,raw_total,view_total,difference
0,operations,2023년 01월,13632.0,13632.0,0.0
1,operations,2023년 02월,12937.0,12937.0,0.0
2,operations,2023년 03월,14407.0,14407.0,0.0
3,operations,2023년 04월,15231.0,15231.0,0.0
4,operations,2023년 05월,16782.0,16782.0,0.0


route_scope_summary (2, 4)


...

monthly_period_parse_check (1, 5)


,rows_count,invalid_period_rows,first_period,last_period,periods
0,9634,0.0,202301,202512,36


route_scope_monthly_comparison (36, 9)


,period_ym,all_passengers,international_passengers,domestic_exception_passengers,unresolved_passengers,all_operations,international_operations,domestic_exception_operations,unresolved_operations
0,202301,2403433.0,2403433.0,0.0,0.0,13632.0,13632.0,0.0,0.0
1,202302,2212589.0,2212589.0,0.0,0.0,12937.0,12937.0,0.0,0.0
2,202303,2256529.0,2256529.0,0.0,0.0,14407.0,14407.0,0.0,0.0
3,202304,2412373.0,2412373.0,0.0,0.0,15231.0,15231.0,0.0,0.0
4,202305,2622755.0,2622755.0,0.0,0.0,16782.0,16781.0,1.0,0.0


,origin_airport_raw,destination_airport_raw,period_raw,period_ym,passengers,operations,origin_airport_std,destination_airport_std,route_scope_std,is_international
0,제주(CJU),인천(ICN),2023년 05월,202305,0,1,제주,인천,domestic,0
1,인천(ICN),제주(CJU),2023년 01월,202301,0,0,인천,제주,domestic,0
2,김포(GMP),간사이(KIX),2023년 01월,202301,16815,100,김포,간사이,international,1
3,김포(GMP),간사이(KIX),2023년 02월,202302,21357,112,김포,간사이,international,1
4,김포(GMP),간사이(KIX),2023년 03월,202303,26967,146,김포,간사이,international,1


국내 공항 참조로 출발·도착 공항이 모두 국내인 행을 분리한다. 괄호·공백·구분자를 제거한 목적지 키를 만들되 원문 이름은 남긴다. 미확정 행은 국제선으로 추정하지 않는다. 행 수와 월별 여객·운항 합계가 raw 세부 행과 일치하는지 확인한 후 국제선만 분석에 사용한다.

## 3. 2026년 1–7월 홀드아웃 노선 자료

In [6]:
holdout_sql=pdx.named_sql('00b_2026_route_holdout.sql')
pdx.apply_ddl(engine,holdout_sql,['create_raw_route_passenger_2026','create_raw_route_operations_2026'])
route_2026=[dict(d,target=d['target']+'_2026') for d in DATASETS[:2]]
manifest_2026=pdx.source_manifest(HOLDOUT_ROOT,route_2026)
display(manifest_2026)
assert manifest_2026.file_count.eq(1).all() and manifest_2026.file_count.sum()==2
loaded_2026=pdx.load_files(engine,HOLDOUT_ROOT,route_2026,'portfolio_2026_route_holdout')
display(loaded_2026)
pdx.apply_ddl(engine,holdout_sql,['create_v_route_monthly_2026'])
route_integrity=pdx.query(engine,holdout_sql,'route_2026_join_integrity_check')
display(route_integrity)
assert int(route_integrity.passenger_detail_rows.iat[0])==int(route_integrity.passenger_view_rows.iat[0])
assert int(route_integrity.operation_detail_rows.iat[0])==int(route_integrity.operation_view_rows.iat[0])
assert int(route_integrity.invalid_period_rows.iat[0])==0 and int(route_integrity.unresolved_scope_rows.iat[0])==0
source_rows={d['target']:sum(len(pdx.prepare_frame(path,d)) for path in HOLDOUT_ROOT.glob(d['pattern'])) for d in route_2026}
db_rows={table:int(pd.read_sql(text(f'SELECT COUNT(*) AS n FROM {table}'),engine).n.iat[0]) for table in source_rows}
display(pd.DataFrame([{'table':table,'Excel_rows':source_rows[table],'DB_rows':db_rows[table]} for table in source_rows]))
assert source_rows==db_rows
display(pd.read_sql(text('SELECT * FROM v_route_monthly_2026 LIMIT 5'),engine))

,target_table,file_count
0,raw_route_passenger_2026,1
1,raw_route_operations_2026,1


,target_table,result,rows
0,raw_route_passenger_2026,skipped,0
1,raw_route_operations_2026,skipped,0


,passenger_detail_rows,passenger_view_rows,operation_detail_rows,operation_view_rows,invalid_period_rows,unresolved_scope_rows
0,2072,2072,2072,2072,0.0,0.0


,table,Excel_rows,DB_rows
0,raw_route_passenger_2026,2079,2079
1,raw_route_operations_2026,2079,2079


,origin_airport_raw,destination_airport_raw,period_raw,period_ym,passengers,operations,origin_airport_std,destination_airport_std,route_scope_std,is_international
0,김포(GMP),간사이(KIX),2026년 01월,202601,52407,279,김포,간사이,international,1
1,김포(GMP),간사이(KIX),2026년 02월,202602,47747,280,김포,간사이,international,1
2,김포(GMP),간사이(KIX),2026년 03월,202603,55812,310,김포,간사이,international,1
3,김포(GMP),간사이(KIX),2026년 04월,202604,50346,300,김포,간사이,international,1
4,김포(GMP),간사이(KIX),2026년 05월,202605,52604,310,김포,간사이,international,1


2026년 1–7월 자료는 2025년 1–7월 <strong>기준월</strong> Rolling Validation에 섞지 않는다. 2026년 1–7월 <strong>대상월</strong> Fixed Test에서는 실제값으로 사용하고, 2026년 7월까지 확인된 값은 8–12월 예측의 입력으로 사용한다.

## 4. 월별 결합 패널과 전후 정합성

In [7]:
model_sql=pdx.named_sql('05_demand_forecast.sql')
pdx.apply_ddl(engine,model_sql,['create_v_model_destination_monthly_all','create_v_demand_destination_profile'])
market=pd.read_sql(text('SELECT period_ym,SUM(passengers) AS passengers FROM v_route_monthly WHERE is_international=TRUE GROUP BY period_ym ORDER BY period_ym'),engine)
dest=pd.read_sql(text('SELECT period_ym,destination_airport_std AS dest,SUM(passengers) AS passengers FROM v_route_monthly WHERE is_international=TRUE GROUP BY period_ym,destination_airport_std'),engine)
panel=pd.read_sql(text('SELECT * FROM v_model_destination_monthly_all'),engine)
profile=pd.read_sql(text('SELECT * FROM v_demand_destination_profile'),engine)
print('market',market.shape,market.period_ym.min(),market.period_ym.max())
print('destination-month',dest.shape,dest.dest.nunique())
print('model panel',panel.shape,panel.period_ym.min(),panel.period_ym.max())
print('universe >=24 months',int(profile.in_modeling_universe.sum()))
display(market.head());display(dest.head());display(panel.head());display(profile.head())
assert len(market)==36 and len(dest)==6493 and dest.dest.nunique()==275
assert int(profile.in_modeling_universe.sum())==169
monthly_parity=dest.groupby('period_ym').passengers.sum().rename('destination_sum').reset_index().merge(market,on='period_ym')
monthly_parity['difference']=monthly_parity.destination_sum-monthly_parity.passengers
assert monthly_parity.difference.eq(0).all()
display(monthly_parity[['period_ym','difference']].head())
display(pdx.query(engine,model_sql,'model_view_parity_signature'))

market (36, 2) 202301 202512
destination-month (6493, 3) 275
model panel (7825, 5) 202301 202607
universe >=24 months 169


,period_ym,passengers
0,202301,2403433.0
1,202302,2212589.0
2,202303,2256529.0
3,202304,2412373.0
4,202305,2622755.0


,period_ym,dest,passengers
0,202301,간사이,201845.0
1,202302,간사이,203627.0
2,202303,간사이,215973.0
3,202304,간사이,205466.0
4,202305,간사이,231953.0


,period_ym,destination_canonical_std,passengers,operations,origin_airport_count
0,202301,간사이,201845.0,1046.0,6
1,202301,과달라하라,0.0,12.0,1
2,202301,괌,41882.0,226.0,2
3,202301,광저우,2343.0,43.0,1
4,202301,구마모도,1785.0,12.0,1


,destination_canonical_std,first_period,last_period,months_present,months_positive,months_2023,months_2024,months_2025,months_2026,total_passengers,q1_2024,q2_2024,q1_2025,q2_2025,sample_band,in_modeling_universe,seasonal_label_2024,seasonal_label_2025
0,간사이,202301,202607,43,43.0,12.0,12.0,12.0,7.0,12553666.0,902096.0,852576.0,914397.0,867574.0,ge36,1,winter,winter
1,과달라하라,202301,202607,43,0.0,12.0,12.0,12.0,7.0,0.0,0.0,0.0,0.0,0.0,ge36,1,flat,flat
2,괌,202301,202607,43,43.0,12.0,12.0,12.0,7.0,1407537.0,130442.0,91707.0,94112.0,81798.0,ge36,1,winter,winter
3,광저우,202301,202607,43,43.0,12.0,12.0,12.0,7.0,1168873.0,70142.0,73302.0,91157.0,97836.0,ge36,1,flat,summer
4,구마모도,202301,202607,43,43.0,12.0,12.0,12.0,7.0,365865.0,14561.0,13644.0,49986.0,36111.0,ge36,1,winter,winter


,period_ym,difference
0,202301,0.0
1,202302,0.0
2,202303,0.0
3,202304,0.0
4,202305,0.0


,rows_,destinations,min_period,max_period,sum_passengers,sum_operations,sum_origin_airport_count
0,7825,286,202301,202607,155291873,906785,11442.0
